In [3]:
import h2o
from h2o.automl import H2OAutoML
import pandas as pd

# 🔥 Start H2O
h2o.init()


Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "23.0.2" 2025-01-21; OpenJDK Runtime Environment Homebrew (build 23.0.2); OpenJDK 64-Bit Server VM Homebrew (build 23.0.2, mixed mode, sharing)
  Starting server from /Users/sandilranasinghe/Work/workshops/datastorm/code/venv/lib/python3.12/site-packages/h2o/backend/bin/h2o.jar
  Ice root: /var/folders/mt/hgtqnnf96rvbcm8f8hlb2m640000gn/T/tmpx99xtc53
  JVM stdout: /var/folders/mt/hgtqnnf96rvbcm8f8hlb2m640000gn/T/tmpx99xtc53/h2o_sandilranasinghe_started_from_python.out
  JVM stderr: /var/folders/mt/hgtqnnf96rvbcm8f8hlb2m640000gn/T/tmpx99xtc53/h2o_sandilranasinghe_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,01 secs
H2O_cluster_timezone:,Asia/Colombo
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.7
H2O_cluster_version_age:,1 month and 2 days
H2O_cluster_name:,H2O_from_python_sandilranasinghe_ynxgl8
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,5.984 Gb
H2O_cluster_total_cores:,10
H2O_cluster_allowed_cores:,10
H2O_cluster_status:,"locked, healthy"


In [4]:
# Load your pre-downloaded dataset
df = pd.read_csv("adult_income.csv")

# Optional: Strip whitespace in categorical columns
for col in df.select_dtypes(include='object'):
    df[col] = df[col].str.strip()

df.head()


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [5]:
# Convert pandas DataFrame to H2OFrame
hf = h2o.H2OFrame(df)

# Convert target to categorical
hf['income'] = hf['income'].asfactor()

# List of predictors
x = [col for col in hf.columns if col != 'income']
y = 'income'


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [6]:
# Run AutoML for 5 minutes or 20 models
aml = H2OAutoML(max_runtime_secs=120,  # 2 minutes
                max_models=12,
                seed=42,
                sort_metric='AUC')

aml.train(x=x, y=y, training_frame=hf)


AutoML progress: |
15:56:38.296: AutoML: XGBoost is not available; skipping it.

███████████████████████████████████████████████████████████████| (done) 100%


Model Details
=============
H2OGradientBoostingEstimator : Gradient Boosting Machine
Model Key: GBM_2_AutoML_1_20250430_155638


Model Summary: 
    number_of_trees    number_of_internal_trees    model_size_in_bytes    min_depth    max_depth    mean_depth    min_leaves    max_leaves    mean_leaves
--  -----------------  --------------------------  ---------------------  -----------  -----------  ------------  ------------  ------------  -------------
    77                 77                          79346                  7            7            7             27            105           73.7922

ModelMetricsBinomial: gbm
** Reported on train data. **

MSE: 0.07499430077021248
RMSE: 0.2738508732325177
LogLoss: 0.2411341520168401
Mean Per-Class Error: 0.14709728167309086
AUC: 0.9500565935467414
AUCPR: 0.8754722916277619
Gini: 0.9001131870934829

Confusion Matrix (Act/Pred) for max f1 @ threshold = 0.40969931872857257
       <=50K    >50K    Error    Rate
-----  -------  ------  -------  ----------------
<=50K  22889    1831    0.0741   (1831.0/24720.0)
>50K   1726     6115    0.2201   (1726.0/7841.0)
Total  24615    7946    0.1092   (3557.0/32561.0)

Maximum Metrics: Maximum metrics at their respective thresholds
metric                       threshold    value     idx
---------------------------  -----------  --------  -----
max f1                       0.409699     0.774688  193
max f2                       0.18209      0.840671  281
max f0point5                 0.636931     0.812294  120
max accuracy                 0.461508     0.89383   176
max precision                0.993579     1         0
max recall                   0.00348869   1         397
max specificity              0.993579     1         0
max absolute_mcc             0.440975     0.703312  183
max min_per_class_accuracy   0.28445      0.871561  239
max mean_per_class_accuracy  0.240371     0.87354   256
max tns                      0.993579     24720     0
max fns                      0.993579     7752      0
max fps                      0.00265949   24720     399
max tps                      0.00348869   7841      397
max tnr                      0.993579     1         0
max fnr                      0.993579     0.988649  0
max fpr                      0.00265949   1         399
max tpr                      0.00348869   1         397

Gains/Lift Table: Avg response rate: 24.08 %, avg score: 24.06 %
group    cumulative_data_fraction    lower_threshold    lift        cumulative_lift    response_rate    score       cumulative_response_rate    cumulative_score    capture_rate    cumulative_capture_rate    gain      cumulative_gain    kolmogorov_smirnov
-------  --------------------------  -----------------  ----------  -----------------  ---------------  ----------  --------------------------  ------------------  --------------  -------------------------  --------  -----------------  --------------------
1        0.010012                    0.990798           4.15266     4.15266            1                0.992248    1                           0.992248            0.0415763       0.0415763                  315.266   315.266            0.0415763
2        0.020024                    0.987563           4.15266     4.15266            1                0.989264    1                           0.990756            0.0415763       0.0831527                  315.266   315.266            0.0831527
3        0.0300052                   0.982923           4.15266     4.15266            1                0.985531    1                           0.989018            0.0414488       0.124601                   315.266   315.266            0.124601
4        0.0400172                   0.97488            4.15266     4.15266            1                0.979322    1                           0.986592            0.0415763       0.166178                   315.266   315.266            0.166178
5        0.0500292                   0.963229           4.15266     4.15266            1        

In [7]:
# Display model leaderboard
lb = aml.leaderboard
lb.head(rows=10)

# Best model
best_model = aml.leader


In [8]:
# Predict on training data (just for demo — normally use a test set)
pred = best_model.predict(hf)

# View predictions
pred.head()

# Performance on training data
perf = best_model.model_performance(hf)
perf.show()


gbm prediction progress: |███████████████████████████████████████████████████████| (done) 100%


ModelMetricsBinomial: gbm
** Reported on test data. **

MSE: 0.07499430089858776
RMSE: 0.27385087346690673
LogLoss: 0.24113415353967813
Mean Per-Class Error: 0.14719532659421536
AUC: 0.9500419879283609
AUCPR: 0.8754373425911695
Gini: 0.9000839758567218

Confusion Matrix (Act/Pred) for max f1 @ threshold = 0.4091232536552397
       <=50K    >50K    Error    Rate
-----  -------  ------  -------  ----------------
<=50K  22881    1839    0.0744   (1839.0/24720.0)
>50K   1725     6116    0.22     (1725.0/7841.0)
Total  24606    7955    0.1095   (3564.0/32561.0)

Maximum Metrics: Maximum metrics at their respective thresholds
metric                       threshold    value     idx
---------------------------  -----------  --------  -----
max f1                       0.409123     0.774373  195
max f2                       0.185697     0.840572  276
max f0point5                 0.642632     0.812051  117
max accuracy                 0.462155     0.89383   177
max precision                0.993896     1         0
max recall                   0.00405631   1         396
max specificity              0.993896     1         0
max absolute_mcc             0.441895     0.703861  184
max min_per_class_accuracy   0.283821     0.871278  239
max mean_per_class_accuracy  0.256464     0.873676  250
max tns                      0.993896     24720     0
max fns                      0.993896     7784      0
max fps                      0.00268349   24720     399
max tps                      0.00405631   7841      396
max tnr                      0.993896     1         0
max fnr                      0.993896     0.992731  0
max fpr                      0.00268349   1         399
max tpr                      0.00405631   1         396

Gains/Lift Table: Avg response rate: 24.08 %, avg score: 24.06 %
group    cumulative_data_fraction    lower_threshold    lift        cumulative_lift    response_rate    score       cumulative_response_rate    cumulative_score    capture_rate    cumulative_capture_rate    gain      cumulative_gain    kolmogorov_smirnov
-------  --------------------------  -----------------  ----------  -----------------  ---------------  ----------  --------------------------  ------------------  --------------  -------------------------  --------  -----------------  --------------------
1        0.010012                    0.990798           4.15266     4.15266            1                0.992248    1                           0.992248            0.0415763       0.0415763                  315.266   315.266            0.0415763
2        0.020024                    0.987563           4.15266     4.15266            1                0.989264    1                           0.990756            0.0415763       0.0831527                  315.266   315.266            0.0831527
3        0.0300052                   0.982923           4.15266     4.15266            1                0.985531    1                           0.989018            0.0414488       0.124601                   315.266   315.266            0.124601
4        0.0400172                   0.97488            4.15266     4.15266            1                0.979322    1                           0.986592            0.0415763       0.166178                   315.266   315.266            0.166178
5        0.0500292                   0.963229           4.15266     4.15266            1                0.969351    1                           0.983142            0.0415763       0.207754                   315.266   315.266            0.207754
6        0.100028                    0.793741           3.87973     4.01623            0.934275         0.872363    0.967148                    0.927769            0.19398         0.401734                   287.973   301.623            0.397406
7        0.150026                    0.653783           3.26754     3.76672            0.786855         0.724226    0.907062                    0.859935            0.163372        0.565106                   226.7

In [9]:
h2o.shutdown(prompt=False)

H2O session _sid_8831 closed.


/var/folders/mt/hgtqnnf96rvbcm8f8hlb2m640000gn/T/ipykernel_13183/1954269801.py:1: H2ODeprecationWarning: Deprecated, use ``h2o.cluster().shutdown()``.
  h2o.shutdown(prompt=False)
